# HW 1 — 생일 문제 (Birthday Problem)

**확률통계 · Topic 1** | 배점 25점 | 개인 과제

---

## ✍️ 제출자 정보 — 먼저 채우세요

| | |
|---|---|
| **학번** | (여기에 작성) |

> 파일명을 **`HW_학번_T01.ipynb`** 로 바꿔서 제출한다. (예: `HW_202512345_T01.ipynb`)
> ⚠️ **제출 방법과 기한은 PLATO · Google Classroom 공지**를 확인한다.

---

## 문제

같은 강의실에 $n$명이 있을 때 **생일이 같은 사람이 적어도 두 명 있을 확률**은 얼마인가?

대부분 "50%가 되려면 180명쯤"이라고 답하지만, 실제로는 훨씬 적은 인원이면 넘어선다.
이 과제는 그 값을 **시뮬레이션과 이론값 두 방법으로** 구하고 비교한다.

> **가정** — 1년은 365일(윤년 무시), 모든 날짜의 출생 확률이 같다.
> 이 가정에 대해서는 문제 4에서 다시 생각한다.

### 할 일

| 문제 | 내용 | 배점 |
|:-:|---|:-:|
| 1 | 시뮬레이션으로 확률 추정 | 8 |
| 2 | 이론값 계산 | 7 |
| 3 | 두 결과를 한 그래프에 | 5 |
| 4 | 해석 (a)(b) 서술 | 5 |

⚠️ **제출 전 `런타임 → 모두 실행`** 으로 출력을 남길 것. 출력이 없으면 −3점.

## Part 0. 준비

이 셀을 먼저 실행한다. **시드는 바꾸지 말 것** — 채점자가 같은 결과를 재현해야 한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

rng = np.random.default_rng(20260302)   # ⚠️ 이 줄은 바꾸지 마세요

N_RANGE = np.arange(2, 61)   # n = 2, 3, ..., 60
N_TRIALS = 10000             # 각 n 마다 반복 횟수 (최소 10,000)

print(f"n 은 {N_RANGE[0]}부터 {N_RANGE[-1]}까지 · 각 n 마다 {N_TRIALS:,}회 반복")

## 문제 1 — 시뮬레이션 (8점)

각 $n$에 대해 **생일이 겹치는 사람이 있을 확률**을 시뮬레이션으로 추정한다.

**한 번의 실험**은 이렇다.
1. $n$명의 생일을 무작위로 만든다 → `rng.integers(0, 365, size=n)`
2. 겹쳤는지 판정한다 → 서로 다른 날짜의 개수가 $n$보다 **작으면** 겹친 것이다
   (`np.unique(...)` 의 길이를 세면 된다)

이것을 `N_TRIALS` 번 반복해 **겹친 비율**을 구하면 그 $n$의 추정 확률이다.

### ✏️ TODO 1 — `has_duplicate` 를 완성하세요

In [ ]:
def has_duplicate(n, rng):
    """n명의 생일을 뽑아 겹치는 사람이 있으면 True."""
    birthdays = rng.integers(0, 365, size=n)

    # TODO 1: 겹쳤는지 판정해 True / False 를 돌려주세요
    #         힌트 - np.unique(birthdays) 의 길이가 n 보다 작으면 겹친 것이다
    return False        # <- 이 줄을 고치세요


sim_prob = []
for n in N_RANGE:
    hits = [has_duplicate(int(n), rng) for _ in range(N_TRIALS)]
    sim_prob.append(np.mean(hits))

sim_prob = np.array(sim_prob)

print("n = 10, 23, 40 일 때 추정값")
for n in (10, 23, 40):
    print(f"  n = {n:>2} : {sim_prob[n - 2]:.4f}")

## 문제 2 — 이론값 (7점)

여사건으로 계산한다. **모두 생일이 다를** 확률을 구해 1에서 빼면 된다.

$$
\mathbb{P}[\text{적어도 두 명 일치}] \;=\; 1 - \frac{365 \times 364 \times \cdots \times (365-n+1)}{365^{\,n}}
$$

분자는 순열 $P(365, n)$ 이다. 랩 노트북 **부록 A** 에서 본 `math.perm(365, n)` 을 쓰면 한 줄이다.

### ✏️ TODO 2 — `theory_prob` 를 완성하세요

In [ ]:
def theory_prob(n):
    """n명 중 생일이 겹치는 사람이 있을 이론적 확률."""
    # TODO 2: 위 식을 코드로 옮기세요
    #         힌트 - 모두 다를 확률 = math.perm(365, n) / 365 ** n
    return 0.0          # <- 이 줄을 고치세요


theo_prob = np.array([theory_prob(int(n)) for n in N_RANGE])

print("n = 10, 23, 40 일 때 이론값")
for n in (10, 23, 40):
    print(f"  n = {n:>2} : {theo_prob[n - 2]:.4f}")

### 두 결과가 얼마나 맞는지 확인 (채우지 않아도 됨)

시뮬레이션이 제대로 됐다면 이론값과 **소수 둘째 자리까지는** 대체로 맞는다.

In [ ]:
gap = np.abs(sim_prob - theo_prob)
print(f"최대 오차 : {gap.max():.4f}")
print(f"평균 오차 : {gap.mean():.4f}")

## 문제 3 — 그래프 (5점)

시뮬레이션과 이론값을 **하나의 그래프에 겹쳐** 그린다.

**반드시 넣을 것**
- 두 곡선 (시뮬레이션은 점 또는 옅은 선, 이론값은 실선)
- $y = 0.5$ 수평 기준선
- **50%를 넘는 최소 $n$** 을 그래프 위에 표시
- 축 이름 · 제목 · 범례

> 📌 라벨은 **영어**로. Colab 에는 한글 폰트가 없어 글자가 □□□ 로 깨진다.

### ✏️ TODO 3 — 두 곡선을 그리세요
### ✏️ TODO 4 — 50%를 넘는 최소 n 을 구해 표시하세요

In [ ]:
plt.figure(figsize=(7, 4))

# TODO 3: 시뮬레이션과 이론값을 겹쳐 그리세요
#         힌트 - plt.plot(N_RANGE, sim_prob, "o", ms=3, alpha=0.5, label="Simulation")
#                plt.plot(N_RANGE, theo_prob, "-", lw=2, label="Theory")


# TODO 4: 이론값이 0.5 를 처음 넘는 n 을 구하세요
#         힌트 - np.argmax(theo_prob > 0.5) 가 그 위치(인덱스)를 준다
cross = 0          # <- 이 줄을 고치세요

plt.axhline(0.5, color="gray", ls="--", lw=1)
if cross:
    plt.axvline(cross, color="red", ls="--", lw=1.2)
    plt.annotate(f"n = {cross}", xy=(cross, 0.5), xytext=(0.45, 0.25),
                 textcoords="axes fraction", color="red", fontweight="bold",
                 arrowprops=dict(arrowstyle="->", color="red"))

plt.xlabel("Number of people (n)")
plt.ylabel("P[at least one shared birthday]")
plt.title("Birthday Problem: simulation vs theory")
plt.ylim(0, 1.02)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"50%를 넘는 최소 n = {cross}")

## 문제 4 — 해석 (5점)

아래 두 물음에 **각각 3문장 이내**로 답한다. 이 셀을 더블클릭해서 바로 쓰면 된다.

---

### (a) 왜 여사건으로 계산하는 것이 더 쉬운가?

"적어도 두 명이 같다"를 직접 세려면 무엇을 해야 하는지와 **비교해서** 설명할 것.

> **답:** (여기에 작성)

---

### (b) "모든 날짜의 출생 확률이 같다"는 가정이 깨지면?

특정 계절에 출생이 몰린다면 일치 확률은 **커질까 작아질까?**
직관적인 이유를 쓸 것. (증명은 요구하지 않는다)

> **답:** (여기에 작성)

---

## ✅ 제출 전 점검

- [ ] 맨 위에 **학번**을 적었다
- [ ] `런타임 → 모두 실행` 으로 **모든 출력이 남아 있다**
- [ ] 그래프에 축 이름 · 제목 · 범례가 있고 **한글이 없다**
- [ ] 문제 4 (a)(b) 를 모두 작성했다
- [ ] 파일명을 **`HW_학번_T01.ipynb`** 로 바꿨다

**제출처와 기한은 PLATO · Google Classroom 공지를 확인한다.**